# **Headpose Estimation with RetinaFace**

Author: Aditya Khadkikar

Credit: https://github.com/fisakhan/Face_Pose/blob/master/pose_detection_retinaface.py

> Because of the files being excessively large in size, outputs are not stored. Run the notebooks to achieve the outputs. You need the necessary libraries in this notebook, and a ZIP folder of all of the images named as `intra-sim.zip`.

In [ ]:
! pip install deepface

In [3]:
# @title
# built-in dependencies
import itertools, pickle
import math, os, glob, cv2
import matplotlib.pyplot as plt

# 3rd party dependencies
import pandas as pd, numpy as np
pd.set_option('display.max_rows', None)

from deepface import DeepFace
from deepface.modules.verification import find_distance, find_threshold
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from PIL import Image
from scipy import stats
from retinaface import RetinaFace

font = cv2.FONT_HERSHEY_SIMPLEX # Text in video
font_size = 0.4
blue = (0, 0, 255)
green = (0,128,0)
red = (255, 0, 0)

Schema of the returned output from `RetinaFace.detect_faces()`

```json
{
    "face_1": {
        "score": 0.9993440508842468,
        "facial_area": [155, 81, 434, 443],
        "landmarks": {
          "right_eye": [257.82974, 209.64787],
          "left_eye": [374.93427, 251.78687],
          "nose": [303.4773, 299.91144],
          "mouth_right": [228.37329, 338.73193],
          "mouth_left": [320.21982, 374.58798]
        }
  }
}
```

In [ ]:
def draw_landmarks(frame, bb, points):
    '''
    Parameters ----------
      frame : TYPE
          RGB image
      bb : TYPE - Array of float64, Size = (5,)
          coordinates of bounding box for the selected face.
      points : TYPE - Array of float32, Size = (10,)
          coordinates of landmarks for the selected faces.
    Returns ------- None.
    '''
    bb = bb.astype(int)
    points = points.astype(int)

    #print(bb)
    #print(points)

    # draw rectangle and landmarks on face
    cv2.rectangle(frame, (bb[0], bb[1]), (bb[2], bb[3]), red, 1)

    cv2.circle(frame, (points[4], points[5]), 2, blue, 2) # nose x,y
    cv2.circle(frame, (points[6], points[7]), 2, blue, 2) # mouth - right x,y
    cv2.circle(frame, (points[0], points[1]), 2, blue, 2) # right eye x,y
    cv2.circle(frame, (points[2], points[3]), 2, blue, 2) # left eye x,y
    cv2.circle(frame, (points[8], points[9]), 2, blue, 2) # mouth - left x,y

In [8]:
# OLD method of computing roll, yaw and pitch (not used later in the project)
def find_roll(points):
    """
    Parameters ---------- points : TYPE - Array of float32, Size = (10,), coordinates of landmarks for the selected faces.
    Returns ------- roll of face.
    """
    re_y = points[1] # right eye y-coord
    le_y = points[3] # left eye y-coord
    return re_y - le_y # difference between y-coords of the right and left eyes of the subject

def find_yaw(points):
    """
    Parameters ---------- points : TYPE - Array of float32, Size = (10,), coordinates of landmarks for the selected faces.
    Returns ------- yaw of face.
    """
    le2n = points[4] - points[2] # nose_x - lefteye_x
    re2n = points[4] - points[0] # nose_x - righteye_x
    return le2n - re2n # difference between the left/right eye-nose distance lines

def find_pitch(points):
    """
    Parameters ---------- points : TYPE - Array of float32, Size = (10,), coordinates of landmarks for the selected faces.
    Returns ------- pitch of the face.
    """
    eye_y = (points[1] + points[3]) / 2 # midpoint of y-coords between the eyes
    mou_y = (points[7] + points[9]) / 2 # midpoint of y-coords between right/left edges of the mouth
    e2n = eye_y - points[5] # 1) difference between (midpoint of eyes) and (nose_y)
    n2m = points[5] - mou_y # 2) difference between (nose_y) to the (midpoint of mouth)
    return e2n / n2m # ratio of the (midpoint_eyes to nose) dist, and (nose to midpoint_mouth) dist

In [9]:
# OPTION 2 (second method of pose estimation, used)
# - coords are w.r.t the return obj of Retinaface.detect_faces()
def find_pose(unflattened_points):
    """
    Parameters ---------- points : TYPE - Array of float32, Size = (10,), coordinates of landmarks for the selected faces.
    Returns ------- Angle, Yaw, Pitch
    """
    LMx = unflattened_points[:,0] # horizontal coordinates of landmarks
    LMy = unflattened_points[:,1] # vertical coordinates of landmarks

    dPx_eyes = max((LMx[1] - LMx[0]), 1) # x-dist between right and left eye
    dPy_eyes = (LMy[1] - LMy[0]) # y-dist between eyes
    angle = np.arctan(dPy_eyes / dPx_eyes) # angle for rotation based on slope eyes

    alpha = np.cos(angle)
    beta = np.sin(angle)

    # rotated landmarks
    LMxr = (alpha * LMx + beta * LMy + (1 - alpha) * LMx[2] / 2 - beta * LMy[2] / 2)
    LMyr = (-beta * LMx + alpha * LMy + beta * LMx[2] / 2 + (1 - alpha) * LMy[2] / 2)

    # average distance between eyes and mouth
    dXtot = (LMxr[1] - LMxr[0] + LMxr[4] - LMxr[3]) / 2
    dYtot = (LMyr[3] - LMyr[0] + LMyr[4] - LMyr[1]) / 2

    # average distance between nose and eyes
    dXnose = (LMxr[1] - LMxr[2] + LMxr[4] - LMxr[2]) / 2
    dYnose = (LMyr[3] - LMyr[2] + LMyr[4] - LMyr[2]) / 2

    # relative rotation
    # 0 degree = frontal, -90 degree = profile right, 90 degree = profile left (from viewer's perspective)
    Xfrontal = (-90+90 / 0.5 * dXnose / dXtot) if dXtot != 0 else 0
    Yfrontal = (-90+90 / 0.5 * dYnose / dYtot) if dYtot != 0 else 0

    Xfrontal = 180 if Xfrontal > 180 else Xfrontal
    Xfrontal = -180 if Xfrontal < -180 else Xfrontal
    Yfrontal = 180 if Yfrontal > 180 else Yfrontal
    Yfrontal = -180 if Yfrontal < -180 else Yfrontal

    return (angle * 180 / np.pi), Xfrontal, Yfrontal # 1st: roll, 2nd: yaw, 3rd: pitch

In [ ]:
! unzip intra-sim.zip

In [ ]:
# @title
results = {}
not_recog = []

for person in glob.glob(f'intra-sim/*'):
  images = glob.glob(f'{person}/*.png')
  for image_path in images:
    # Read image (PNG) and convert to BGR for OpenCV drawing
    try:
        pil = Image.open(image_path).convert("RGB")
        frame = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        rets = True
    except Exception as e:
        print("Error reading image:", e)
        frame = None
        rets = False
        #not_recog += [image_path]

    if not rets or frame is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
        #not_recog += [image_path]

    # Use the loaded image (do not flip by default; uncomment if you want)
    image_rgb = frame.copy()  # image in BGR order for OpenCV drawing

    try:
      # face detection
      # - if multiple faces, the output is sorted in descending order, with
      #   the first idx's having the highest confidence face obj.
      detection = RetinaFace.detect_faces(image_rgb)

      ## If multiple subjects in the image, just identify by num of bounding boxes -->
      print(image_path, list(detection.keys()))

      # if > 1: take the face with the largest confidence
      # but that is not the used solution. best to manually label.
      if len(list(detection.keys())) > 1:
        print("manually label, multiple faces present here!")
        not_recog += [image_path]

      # the actual primary subject's face could have slightly lower facial confidence score,
      # and model can mistakenly select a background person.
      print([f'{key}: {detection[key]['score']}' for key in list(detection.keys())])

      bbs = np.array(detection[list(detection.keys())[0]]['facial_area'])
      lmarks_unflattened = np.array(list(detection[list(detection.keys())[0]]['landmarks'].values()))
      lmarks = lmarks_unflattened.flatten()
    except Exception as e:
      print(e)
      bbs, lmarks, lmarks_unflattened = [], [], []
      cv2.putText(image_rgb, 'no face detected', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 2)

    # if at least one face is detected
    if len(bbs) > 0:
        # process only one face (center ?) if multiple faces detected
        draw_landmarks(image_rgb, bbs, lmarks)  # draw landmarks and bbox
        roll, yaw, pitch = find_pose(lmarks_unflattened)
        results[image_path] = {
            "person": person,
            "roll": roll,
            "yaw": yaw,
            "pitch": pitch
        }
        print("roll", roll, " -- yaw", yaw, " -- pitch", pitch)
    else:
        cv2.putText(image_rgb, 'no face detected', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 2)

    # Display inline in Jupyter (convert BGR -> RGB for correct colors)
    try:
      plt.figure(
          figsize=(5, 3)
      )
      plt.imshow(cv2.cvtColor(plot(image_rgb, bbs), cv2.COLOR_BGR2RGB))
      plt.axis('off')
      plt.show()
    except:
      plt.imshow(cv2.cvtColor(image_rgb, cv2.COLOR_BGR2RGB))
      plt.axis('off')
      plt.show()

print("images to manually label:", len(not_recog))

In [12]:
print(not_recog)

['intra-sim/Gray_Davis_/7.png', 'intra-sim/Gray_Davis_/17.png', 'intra-sim/Gray_Davis_/19.png', 'intra-sim/Gray_Davis_/5.png', 'intra-sim/Gray_Davis_/6.png', 'intra-sim/Paul_Bremer_/13.png', 'intra-sim/Paul_Bremer_/1.png', 'intra-sim/Junichiro_Koizumi_/6.png', 'intra-sim/Tom_Daschle_/15.png', 'intra-sim/Tom_Daschle_/18.png', 'intra-sim/Atal_Bihari_Vajpayee_/13.png', 'intra-sim/Atal_Bihari_Vajpayee_/7.png', 'intra-sim/Atal_Bihari_Vajpayee_/9.png', 'intra-sim/Atal_Bihari_Vajpayee_/19.png', 'intra-sim/John_Negroponte_/15.png', 'intra-sim/John_Negroponte_/13.png', 'intra-sim/John_Negroponte_/7.png', 'intra-sim/John_Negroponte_/17.png', 'intra-sim/John_Negroponte_/12.png', 'intra-sim/John_Negroponte_/2.png', 'intra-sim/John_Negroponte_/5.png', 'intra-sim/Mahmoud_Abbas_/13.png', 'intra-sim/Mahmoud_Abbas_/9.png', 'intra-sim/Mahmoud_Abbas_/2.png', 'intra-sim/Mahmoud_Abbas_/5.png', 'intra-sim/Michael_Bloomberg_/1.png', 'intra-sim/Michael_Bloomberg_/7.png', 'intra-sim/Michael_Bloomberg_/18.png',

In [ ]:
print(results)

In [14]:
with open('retinaface-headposes.pickle', 'wb') as handle:
    pickle.dump(results, handle, protocol=pickle.HIGHEST_PROTOCOL)